In [9]:
!pip install openai-whisper --quiet

import whisper
import json
!pip install pydub --quiet

model_whisper = whisper.load_model("base")

In [15]:
video_path = "../j_n.mov"

In [16]:
import os
from pydub import AudioSegment
import whisper

def transcribe_windows(video_path, model_whisper, chunk_length_ms=3000):
    
    audio_path = "temp_full_audio.wav"
    os.system(f"ffmpeg -i {video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {audio_path} -y -q:a 0")
    
    sound = AudioSegment.from_wav(audio_path)
    total_duration_ms = len(sound)
    
    participant_narration_fixed = []
    
    for i, start_ms in enumerate(range(0, total_duration_ms, chunk_length_ms)):
        end_ms = min(start_ms + chunk_length_ms, total_duration_ms)
        
        chunk_audio = sound[start_ms:end_ms]
        chunk_filename = f"temp_chunk_{i}.wav"
        chunk_audio.export(chunk_filename, format="wav")
        
        start_sec = start_ms / 1000.0
        end_sec = end_ms / 1000.0
        
        result = model_whisper.transcribe(chunk_filename, temperature=0.0)
        text = result["text"].strip()
        
        participant_narration_fixed.append({
            "start": start_sec,
            "end": end_sec,
            "text": text
        })
        
        os.remove(chunk_filename)
        
    os.remove(audio_path)    
    return participant_narration_fixed

**using wisper from open ai model to convert video first to audio then to text**

In [17]:
fix_chunk = transcribe_windows(video_path,model_whisper)

ffmpeg version 5.1.9 Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 11 (GCC)
  configuration: --prefix=/usr --bindir=/usr/bin --datadir=/usr/share/ffmpeg --docdir=/usr/share/doc/ffmpeg --incdir=/usr/include/ffmpeg --libdir=/usr/lib64 --mandir=/usr/share/man --arch=x86_64 --optflags='-O2 -flto=auto -ffat-lto-objects -fexceptions -g -grecord-gcc-switches -pipe -Wall -Werror=format-security -Wp,-D_FORTIFY_SOURCE=2 -Wp,-D_GLIBCXX_ASSERTIONS -specs=/usr/lib/rpm/redhat/redhat-hardened-cc1 -fstack-protector-strong -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -m64 -march=x86-64-v2 -mtune=generic -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection' --extra-ldflags='-Wl,-z,relro -Wl,--as-needed -Wl,-z,now -specs=/usr/lib/rpm/redhat/redhat-hardened-ld -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 ' --extra-cflags=' -I/usr/include/rav1e' --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libvo-amrwbenc --enable-version3 --enable-bzlib --disable-crysta

In [20]:
print(len(fix_chunk))
fix_chunk[10:20]

117


[{'start': 30.0, 'end': 33.0, 'text': 'from the clapping, turning.'},
 {'start': 33.0, 'end': 36.0, 'text': 'the furniture, putting to the desk.'},
 {'start': 36.0, 'end': 39.0, 'text': 'moving things around on the desk.'},
 {'start': 39.0,
  'end': 42.0,
  'text': 'now participants writing on the keyboard.'},
 {'start': 42.0, 'end': 45.0, 'text': ''},
 {'start': 45.0, 'end': 48.0, 'text': 'still writing on the keyboard'},
 {'start': 48.0, 'end': 51.0, 'text': 'Good.'},
 {'start': 51.0, 'end': 54.0, 'text': ''},
 {'start': 54.0, 'end': 57.0, 'text': ''},
 {'start': 57.0, 'end': 60.0, 'text': ''}]

In [21]:
import os
import json
def save_load(path:str,config = None ,save=False):
    path_file = './' + path + '.json'
    processed_data = {}

    # path_file = "/content/drive/MyDrive/data/" + path + '.json'
    

    if save:
        with open(path_file,'w',encoding='utf-8') as f :
            json.dump(config,f,ensure_ascii=False, indent=4)
    else:
        if os.path.exists(path_file):
            with open(path_file, 'r', encoding='utf-8') as f :
                processed_data = json.load(f)
                return processed_data

**save and load**

In [24]:
final_j = save_load('../norm_data2_obj')

In [26]:
final_j[4]

{'chunk_id': 4,
 'time': '12-15',
 'cameras_data': {'gopro': {'processed_frame': 1050, 'status': 'success'}},
 'objects': ['MONITOR', 'KEYBOARD', 'PACKAGE'],
 'action_probs': {}}

In [29]:
def merge_video_and_text_lists(video_list, text_list):
    combined_data = []
    
    for v_chunk, t_chunk in zip(video_list, text_list):
        merged_chunk = v_chunk.copy()
        
        merged_chunk["participant_text"] = t_chunk.get("text", "")
        
        combined_data.append(merged_chunk)
        
    return combined_data

### frist input the obj file contain all information from previos level ( object detection and also hand action )
### seouncd input  , the naration 

In [ ]:

final_dataset = merge_video_and_text_lists(final_j, fix_chunk)


In [33]:
final_dataset[7:10]

[{'chunk_id': 7,
  'time': '21-24',
  'cameras_data': {'gopro': {'processed_frame': 1275, 'status': 'success'}},
  'objects': ['PEN', 'MONITOR', 'KEYBOARD'],
  'action_probs': {},
  'participant_text': ''},
 {'chunk_id': 8,
  'time': '24-27',
  'cameras_data': {'gopro': {'processed_frame': 1350, 'status': 'success'}},
  'objects': ['PEN', 'MONITOR', 'KEYBOARD'],
  'action_probs': {},
  'participant_text': 'participants to'},
 {'chunk_id': 9,
  'time': '27-30',
  'cameras_data': {'gopro': {'processed_frame': 1425, 'status': 'success'}},
  'objects': ['PEN', 'MONITOR', 'KEYBOARD'],
  'action_probs': {},
  'participant_text': 'is sitting still as participating.'}]

In [41]:
def propagate_narrations(processed_data):
    
    last_seen_text = ""
    last_seen_objects = []
    
    for chunk in processed_data:
        current_text = chunk.get("participant_text", "").strip()
        current_objs = chunk.get("objects", [])
        # hand_motion = chunk.get("hand_motion", "IDLE")
        
        if current_text != "":
            last_seen_text = current_text
            last_seen_objects = current_objs
            
        else:
            if last_seen_text != "":
                has_object_continuity = any(obj in current_objs for obj in last_seen_objects)
                
                if has_object_continuity or hand_motion in ["GRASP_OBJECT", "ACTIVE"]:
                    chunk["participant_text"] = f"[Propagated] {last_seen_text}"
                else:
                    last_seen_text = ""
                    last_seen_objects = []

   
    return processed_data

**duplicatie participant text to previous upcoming chunk (because of for exmaple at time 10 - 20 the participant typing with keybord but only at frist 1-2 secound of thistime the naration say that and the rest of thoese empty )**

In [42]:
final_dataset_with_propagate = propagate_narrations(final_dataset)

In [44]:
final_dataset_with_propagate[5:10]

[{'chunk_id': 5,
  'time': '15-18',
  'cameras_data': {'gopro': {'processed_frame': 1125, 'status': 'success'}},
  'objects': ['PEN', 'MONITOR', 'KEYBOARD'],
  'action_probs': {},
  'participant_text': '[Propagated] sitting on the chair.'},
 {'chunk_id': 6,
  'time': '18-21',
  'cameras_data': {'gopro': {'processed_frame': 1200, 'status': 'success'}},
  'objects': ['PEN', 'MONITOR', 'KEYBOARD'],
  'action_probs': {},
  'participant_text': '[Propagated] sitting on the chair.'},
 {'chunk_id': 7,
  'time': '21-24',
  'cameras_data': {'gopro': {'processed_frame': 1275, 'status': 'success'}},
  'objects': ['PEN', 'MONITOR', 'KEYBOARD'],
  'action_probs': {},
  'participant_text': '[Propagated] sitting on the chair.'},
 {'chunk_id': 8,
  'time': '24-27',
  'cameras_data': {'gopro': {'processed_frame': 1350, 'status': 'success'}},
  'objects': ['PEN', 'MONITOR', 'KEYBOARD'],
  'action_probs': {},
  'participant_text': 'participants to'},
 {'chunk_id': 9,
  'time': '27-30',
  'cameras_data': {

In [46]:
file_name = "./fd_silent_fix.json"

with open(file_name, "w", encoding="utf-8") as f:
    json.dump(final_dataset_with_propagate, f, indent=4, ensure_ascii=False)
